# Exploratory Data Analysis — BSL Word Transformer

Answers the handbook's five EDA questions from `data/metadata.csv` and the cached
landmark `.npz` files, then creates the organisation-grouped train/val split.

1. How many clips per word (class balance)?
2. How long are clips relative to the fixed 64-frame model input?
3. How often do hands drop out of MediaPipe tracking, per source?
4. What do the extracted skeletons actually look like?
5. How are clips distributed across source organisations (the split grouping unit)?

Requires `python -m src.download`, `python -m src.extract` to have been run
(see README workflow steps 1-3). Cells raise a clear `FileNotFoundError` otherwise.

In [ ]:
from pathlib import Path
import sys
import json

REPO = Path.cwd().resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = REPO / 'data'
META_CSV = DATA / 'metadata.csv'
LANDMARKS = DATA / 'landmarks'

if not META_CSV.exists():
    raise FileNotFoundError(
        f'{META_CSV} not found - run `python -m src.download` first (README workflow step 2).')

meta = pd.read_csv(META_CSV)
print(f'{len(meta)} clips, {meta["word"].nunique()} words, '
      f'{meta["organisation"].nunique()} organisations')
meta.head()

## Q1 — Clips per word (class balance)

In [ ]:
counts = meta['word'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(14, 4))
counts.plot.bar(ax=ax, color='steelblue')
ax.axhline(counts.mean(), color='grey', ls='--', label=f'mean = {counts.mean():.1f}')
ax.set_ylabel('clips')
ax.set_title('Clips per word')
ax.legend()
plt.tight_layout()
plt.show()
counts.describe()

## Q2 — Clip duration distribution vs the fixed 64-frame input

Frame counts are `duration_s * fps`. The red line marks the 64-frame budget every
clip is linearly resampled to; clips far above it lose temporal detail, clips
below it are stretched.

In [ ]:
frames = (meta['duration_s'] * meta['fps']).dropna()
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(frames, bins=40, color='steelblue', edgecolor='white')
ax.axvline(64, color='crimson', ls='--', label='64-frame model input')
ax.set_xlabel('frames per clip')
ax.set_ylabel('count')
ax.set_title('Clip length distribution vs fixed 64-frame resampling')
ax.legend()
plt.tight_layout()
plt.show()
meta['duration_s'].describe()

## Q3 — Hand/face dropout rate per source

Fraction of frames where MediaPipe failed to detect each block, from the cached
`presence` arrays (columns: left hand, right hand, face). High hand dropout is
the main data-quality risk for a pose-based pipeline.

In [ ]:
if not LANDMARKS.exists():
    raise FileNotFoundError(
        f'{LANDMARKS} not found - run '
        '`python -m src.extract --videos data/raw_videos --out data/landmarks` first.')

rows = []
for _, r in meta.iterrows():
    npz_path = LANDMARKS / r['word'] / f"{r['clip_id']}.npz"
    if not npz_path.exists():
        continue
    presence = np.load(npz_path)['presence']  # (T, 3): left hand, right hand, face
    rows.append({
        'source': r['source'],
        'clip_id': r['clip_id'],
        'left_hand_dropout': 1.0 - presence[:, 0].mean(),
        'right_hand_dropout': 1.0 - presence[:, 1].mean(),
        'face_dropout': 1.0 - presence[:, 2].mean(),
    })
drop = pd.DataFrame(rows)
print(f'presence loaded for {len(drop)} cached clips')
drop.groupby('source')[['left_hand_dropout', 'right_hand_dropout',
                        'face_dropout']].mean().round(3)

## Q4 — Skeleton overlays for sample clips

Mid-clip frames in raw MediaPipe image coordinates (y-axis inverted so figures are
upright). Blocks: pose (blue), left hand (orange), right hand (green), mouth (red).

In [ ]:
from src.landmarks import POSE_SLICE, LEFT_HAND_SLICE, RIGHT_HAND_SLICE, MOUTH_SLICE

sample = meta.groupby('word').head(1).sample(
    n=min(6, meta['word'].nunique()), random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (_, r) in zip(axes.flat, sample.iterrows()):
    npz_path = LANDMARKS / r['word'] / f"{r['clip_id']}.npz"
    if not npz_path.exists():
        ax.set_title(f"{r['clip_id']} (no cache)")
        ax.axis('off')
        continue
    seq = np.load(npz_path)['landmarks']  # (T, 105, 3), raw image coords
    mid = seq[len(seq) // 2]
    for sl, colour in [(POSE_SLICE, 'tab:blue'), (LEFT_HAND_SLICE, 'tab:orange'),
                       (RIGHT_HAND_SLICE, 'tab:green'), (MOUTH_SLICE, 'tab:red')]:
        ax.scatter(mid[sl, 0], mid[sl, 1], s=8, c=colour)
    ax.invert_yaxis()
    ax.set_aspect('equal')
    ax.set_title(r['word'], fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('Mid-clip landmark frames (pose / left hand / right hand / mouth)')
plt.tight_layout()
plt.show()

## Q5 — Source organisation distribution

Splits are grouped by organisation so no organisation (and hence, plausibly, no
signer or studio style) appears in both train and val.

In [ ]:
org = (meta.groupby('organisation')
           .agg(clips=('clip_id', 'count'), words=('word', 'nunique'))
           .sort_values('clips', ascending=False))
fig, ax = plt.subplots(figsize=(8, 3))
org['clips'].plot.bar(ax=ax, color='steelblue')
ax.set_ylabel('clips')
ax.set_title('Clips per source organisation (split grouping unit)')
plt.tight_layout()
plt.show()
org

## Augmentation sanity visual

Original vs spatially augmented skeleton on the same normalised clip — the
transform must be label-preserving (rotation/scale/translate/jitter, not a
different sign).

In [ ]:
from src.augment import spatial_augment
from src.dataset import load_clip, resample_sequence

r = sample.iloc[0]
npz_path = LANDMARKS / r['word'] / f"{r['clip_id']}.npz"
seq = resample_sequence(load_clip(str(npz_path)), 64)
rng = np.random.default_rng(42)
aug = spatial_augment(seq, rng)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, s, title in [(axes[0], seq, 'normalised'), (axes[1], aug, 'augmented')]:
    frame = s[32]
    ax.scatter(frame[:, 0], frame[:, 1], s=8, c='steelblue')
    ax.invert_yaxis()
    ax.set_aspect('equal')
    ax.set_title(f"{r['word']} - {title}")
plt.tight_layout()
plt.show()

## Create train/val splits

Greedy organisation-grouped assignment targeting ~80/20 by clip count
(`src.dataset.make_splits`); writes `data/splits.json` and prints a summary.

In [ ]:
from src.dataset import make_splits

splits = make_splits(str(META_CSV), str(DATA / 'vocabulary.csv'),
                     val_frac=0.2, seed=42)
out_path = DATA / 'splits.json'
with open(out_path, 'w') as f:
    json.dump(splits, f, indent=2)

n_train, n_val = len(splits['train']), len(splits['val'])
print(f'wrote {out_path}')
print(f'classes: {len(splits["label_list"])}')
print(f'train clips: {n_train}   val clips: {n_val} '
      f'({n_val / (n_train + n_val):.1%} val)')
print(f'grouped by: {splits["group_by"]}   seed: {splits["seed"]}')